# 06 – Outputs & Visualisations

Generate the full suite of team-level and player-level metrics and
portfolio-quality visualisations across all three competitions
(Euro 2020, World Cup 2022, Euro 2024).


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt

from src.config import COMPETITIONS
from src.data.loader import load_competition
from src.features.buildup import filter_buildup
from src.features.line_detection import detect_opponent_lines
from src.features.receiver import detect_receiver_candidates
from src.features.lane import compute_lane_features
from src.features.findability import compute_findability
from src.metrics.outputs import (
    compute_team_metrics,
    compute_player_metrics,
    merge_event_scores,
)
from src.viz.plots import (
    plot_team_comparison_bar,
    plot_availability_scatter,
    plot_team_heatmap,
    plot_missed_opportunity,
)


## Load all three competitions

This cell takes several minutes to run the full pipeline.
Progress is logged to stderr.


In [ ]:
all_merged, all_candidates_scored = [], []

for comp in COMPETITIONS:
    print(f'\n=== {comp["name"]} ===')
    events, frames, lineups, matches = load_competition(
        competition_id=comp['competition_id'],
        season_id=comp['season_id'],
    )
    buildup_df    = filter_buildup(events)
    line_df       = detect_opponent_lines(frames, buildup_df)
    receivers_df  = detect_receiver_candidates(frames, buildup_df, line_df)
    candidates_df = compute_lane_features(receivers_df, frames, buildup_df)
    event_scores, cands_scored = compute_findability(candidates_df)
    merged        = merge_event_scores(buildup_df, event_scores, line_df)
    merged['competition_name'] = comp['name']
    cands_scored['competition_name'] = comp['name']
    all_merged.append(merged)
    all_candidates_scored.append(cands_scored)

merged_all = pd.concat(all_merged, ignore_index=True)
cands_all  = pd.concat(all_candidates_scored, ignore_index=True)
print(f'\nTotal build-up events: {len(merged_all):,}')


## Team metrics (all competitions)


In [ ]:
team_metrics  = compute_team_metrics(merged_all)
player_metrics = compute_player_metrics(merged_all, cands_all)

display(team_metrics[['competition_name','team','between_lines_availability_rate',
                       'avg_line_gap','total_buildup_events']].head(30))


## Team comparison bar chart


In [ ]:
fig = plot_team_comparison_bar(team_metrics, top_n=25)
plt.show()


## Player availability scatter (top receivers)


In [ ]:
fig = plot_availability_scatter(player_metrics, min_appearances=15)
plt.show()


## Team heatmap example (pick a team)


In [ ]:
# Change to any team in the dataset
team_name = team_metrics.iloc[0]['team']  # top availability team
fig = plot_team_heatmap(merged_all, team=team_name)
plt.show()


## Missed opportunity story plot


In [ ]:
# Find a rich example: findable option but in advanced build-up phase
euro2020_merged = merged_all[merged_all['competition_name'] == 'UEFA Euro 2020']
candidates_euro2020 = cands_all[cands_all['competition_name'] == 'UEFA Euro 2020']

rich_events = euro2020_merged[
    (euro2020_merged['findable_option_available'] == 1) &
    (euro2020_merged['possession_phase'] == 'advanced_buildup')
]['id'].dropna()

# Re-load Euro 2020 frames for plotting
_, frames_e2020, _, _ = load_competition(competition_id=55, season_id=43)
events_with_frames = set(frames_e2020['event_id'].unique())
plottable = [eid for eid in rich_events if eid in events_with_frames]

if plottable:
    buildup_euro = filter_buildup(
        __import__('src.data.loader', fromlist=['load_competition']).load_competition(
            competition_id=55, season_id=43
        )[0]
    )
    line_euro = detect_opponent_lines(frames_e2020, buildup_euro)
    fig = plot_missed_opportunity(
        event_id=plottable[0],
        frames_df=frames_e2020,
        buildup_df=buildup_euro,
        line_df=line_euro,
        candidates_scored_df=candidates_euro2020,
    )
    plt.show()


## Save outputs


In [ ]:
import os
os.makedirs('../outputs', exist_ok=True)
team_metrics.to_csv('../outputs/team_metrics.csv', index=False)
player_metrics.to_csv('../outputs/player_metrics.csv', index=False)
merged_all.to_csv('../outputs/merged_analysis.csv', index=False)
print('Saved team_metrics.csv, player_metrics.csv, merged_analysis.csv')
